In [1]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Power analysis
from statsmodels.stats.power import NormalIndPower

In [3]:
#Exercise 1: Calculating Required Sample Size
power_analysis = NormalIndPower()

effect_size = 0.3
alpha = 0.05
power = 0.8

sample_size = power_analysis.solve_power(effect_size=effect_size,
                                         alpha=alpha,
                                         power=power,
                                         alternative='two-sided')

print(f"Required sample size per group: {sample_size:,}")

Required sample size per group: 174.41912242947043


In [5]:
# Exercise 2: Understanding the Relationship Between Effect Size and Sample Size

power_analysis = NormalIndPower()
effect_sizes = [0.2, 0.4, 0.5]
results = []

for es in effect_sizes:
    n = power_analysis.solve_power(
        effect_size=es, alpha=0.05, power=0.8, alternative='two-sided'
    )
    results.append({
        'Effect Size': es,
        'Sample Size Per Group': int(np.ceil(n))
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))


 Effect Size  Sample Size Per Group
         0.2                    393
         0.4                     99
         0.5                     63


1. How does the sample size change?
The required sample size decreases significantly as the effect size increases. This relationship is not linear; it is an inverse square relationship, meaning that to detect an effect that is twice as small, you actually need four times as much data.

2. Why does this happen?
It comes down to distinguishing the "Signal" from the "Noise":

Small Effects = High Noise: When the change you are looking for is tiny (e.g., a 0.5% increase), it is easily buried by the natural, random fluctuations in user behavior. To prove the change is real and not just a "lucky day," you need a massive amount of data to smooth out the noise and make the signal visible.

Large Effects = Clear Signal: If your new subject line is a "game changer" that doubles your open rate, the difference between the two groups will be obvious almost immediately. Even with a small group of users, the gap between Version A and Version B will be so large that it is mathematically unlikely to be a coincidence.

Practical Takeaway:
If you have a small audience (low traffic), you should only test "bold" changes that you expect will have a large effect. If you have millions of users, you can afford to test "incremental" changes with small effects because you have the statistical volume to detect them.

In [6]:
# Exercise 3: Exploring the Impact of Statistical Power

power_levels = [0.7, 0.8, 0.9]
power_results = []

for pwr in power_levels:
    n = power_analysis.solve_power(
        effect_size=0.2, alpha=0.05, power=pwr, alternative='two-sided'
    )
    power_results.append({
        'Power': pwr,
        'Sample Size Per Group': int(np.ceil(n)),
        'Total Sample Size': int(np.ceil(n)) * 2,
        'Miss Rate': f"{(1-pwr)*100:.0f}%"
    })

df_power = pd.DataFrame(power_results)
print(df_power.to_string(index=False))

 Power  Sample Size Per Group  Total Sample Size Miss Rate
   0.7                    309                618       30%
   0.8                    393                786       20%
   0.9                    526               1052       10%


1. How does the required sample size change?As you increase the Power, the required sample size increases. Specifically, jumping from an 80% to a 90% chance of detecting an effect requires a significantly larger boost in data (about 33% more users in this example) compared to the jump from 70% to 80%.2. Why is this understanding important?Understanding the trade-off between power and sample size is crucial for three main reasons:Risk Management (The Type II Error): Statistical power is $1 - \beta$. If your power is 0.8, you have a 20% chance of a False Negative (missing a winning idea). If the change you are testing is expensive to implement or very important for the company, you might want a power of 0.9 to be "safer," even if it means running the test longer.Resource Allocation: Data isn't free. Running a test longer costs time and prevents you from testing other ideas. If you have limited traffic, you might "settle" for a power of 0.7 to get results faster, accepting a higher risk of missing small improvements.Avoiding "Underpowered" Tests: The biggest mistake in A/B testing is running a test with a tiny sample size (low power). If your power is only 0.2, you have an 80% chance of seeing "no significant difference" even if your new design is actually better. This leads teams to wrongly abandon good ideas.Summary: Think of Power as the sensitivity of your "microscope." If you want to be very sure you see even the smallest germs (effects), you need a more powerful (larger) microscope. In A/B testing, that "power" comes from having more users.

In [ ]:
# Exercise 4: Implementing Sequential Testing

1. Defining Stopping Criteria
In standard A/B testing, "peeking" at the data and stopping early as soon as $p < 0.05$ is a major mistake—it leads to a massive increase in False Positives. To do this correctly, we must use an Alpha Spending Function (like the O'Brien-Fleming or Pocock boundaries).Your stopping criteria should be:
- The Adjusted Significance Threshold: Instead of a flat $0.05$, you set much stricter targets for early weeks. For example, in week 1, you might only stop if $p < 0.005$.
- Minimum Sample Size: You must reach a "burn-in" period (e.g., at least 20% of your total calculated sample) before even looking at the results.

2. How to Implement Sequential TestingTo implement this in your scenario, you would follow these steps:
- Calculate the Total Sample: Determine the total $N$ required (just like in Exercise 1).Determine Interim Analysis Points: Decide how many times you will "peek" (e.g., once a week for 4 weeks).
- Apply a Correction: Use a method like Group Sequential Design. If you plan to check 3 times, you don't use $0.05$ at each check. You might use the Bonferroni correction ($0.05 / 3 \approx 0.016$) or more sophisticated boundaries that "spend" your alpha gradually.
- Sequential Monitoring: Compare the observed p-value at each week against your adjusted threshold for that specific week.

3. What to do at Week 3 ($p = 0.02$)?
This is the "moment of truth." Whether you stop or continue depends entirely on which boundary you chose:
- Scenario A (Traditional/Fixed Test): If you didn't plan for sequential testing, you must keep going. A $p = 0.02$ at an interim stage is not yet "official" because the risk of a false positive is too high.
- Scenario B (Pocock Boundary): If your adjusted threshold for Week 3 was set at $0.015$, then $0.02$ is not significant yet. You must continue to Week 4.
- Scenario C (Flexible/Bayesian): If you are using a Bayesian approach and your "Probability of Version B being better" is $> 99\%$, you might decide to stop if the "Expected Loss" of being wrong is low enough for your business.The Verdict: In a strict Frequentist framework using O'Brien-Fleming, $0.02$ at Week 3 is often not enough to stop early. You typically want to see a very "heavy" signal early on to justify stopping.

In [7]:
# Exercise 5: Applying Bayesian A/B Testing


1. Setting up your Prior Belief
In Bayesian statistics, the Prior represents what you know before the experiment starts.
- Choice of Distribution: For engagement rates (proportions), we typically use a Beta distribution, defined by two parameters: $\alpha$ (successes) and $\beta$ (failures).
- The "Uninformed" Prior: If you believe there is a 50/50 chance of improvement, you might use a Flat Prior (Beta(1,1)). This is like saying, "I have no idea what will happen; I’ll let the data do all the talking.
- "The "Informed" Prior: If you have historical data showing that most new features only have a small impact, you might set a stronger prior centered around your current engagement rate. This "stiffens" the test, requiring more data to change your mind.

2. Influencing the Decision with the Posterior
The Posterior is the result of combining your Prior with the Evidence (the new data collected).

- Interpreting the 65% Probability: In a Bayesian framework, this means there is a 65% chance that B > A.

- Decision Rule: Unlike the p-value (which is "significant" or "not"), Bayesian decisions are based on Risk and Thresholds. Most companies set a threshold (e.g., 95% probability of being better) before launching.

- The Outcome: At 65%, the "signal" is positive but weak. You haven't reached a high enough certainty to declare a winner. However, you have moved from your initial 50% "toss-up" to a more leaning stance toward Version B.

3. What if the Posterior was only 55%?
If the probability is only 55% after collecting a significant amount of data, you find yourself in the "Indifference Zone." Here is what you would do:

- Continue Testing: If you haven't reached your maximum sample size, keep the test running. The probability might climb as more data reduces the uncertainty.

- Analyze the "Expected Loss": Bayesian testing often looks at "If I am wrong and I switch to B, how much engagement do I stand to lose?" If the probability is 55% but the potential loss is tiny, you might switch anyway. If the potential loss is huge, you stay with the Control.

- Declare a "Draw": If the probability stays near 50-55% for a long time, it means the feature has no meaningful impact. You should stop the test, discard the feature, and move on to a new hypothesis.

In [ ]:
# Exercise 6: Implementing Adaptive Experimentation


1. Adjusting Traffic Allocation: The "Exploit vs. Explore" Balance
After Week 1, since Layout C is performing best, you would implement an algorithm to shift traffic. Two common strategies are:

Epsilon-Greedy Strategy: You keep a small percentage (e.g., 10%) of traffic distributed randomly to "Explore" (keep testing A and B to see if they improve), while sending the remaining 90% to "Exploit" the current leader, Layout C.

Thompson Sampling (Bayesian): You use the probability that each layout is the best. If the math says there is a 70% chance C is the winner, a 20% chance for B, and a 10% for A, you allocate exactly that much traffic to each. Layout C now gets 70% of your visitors.

2. Continued Adaptation in Following Weeks
The experiment becomes a "living" process:

Continuous Updates: Every day (or hour), the algorithm recalculates the performance. If Layout B suddenly starts performing better (perhaps because it's better for weekend shoppers), the algorithm will automatically divert more traffic back to B.

Starving the Losers: Over time, the "losing" layouts (A and B) will receive less and less traffic, eventually dropping to near 0%.

Dynamic Stopping: Instead of a hard end date, the test "ends" when one layout captures nearly 100% of the traffic, or when the "regret" (the potential engagement lost by not being on the best version) becomes negligible.



Novelty Effect -> Users might click Layout C just because it’s "new," but the interest fades after a few days.	-> Use a "burn-in" period. Don't start shifting traffic until you have a minimum baseline of data (e.g., after the first full week).

Delayed Feedback	-> If your goal is "purchase" but it takes users 3 days to decide, your daily data will be misleading.	-> Use intermediate metrics (like "add to cart") that happen faster to guide the early adaptation.

Seasonality/Context	-> Layout C might be great on Monday but terrible on Friday.	-> Use Contextual Bandits, which take external factors (day of week, user location) into account when allocating traffic.